# Train results — R0

**Strategy R0: whole-plan regeneration (baseline)**

This notebook reproduces the `R0` column of the **Train results** tab in `Results.xlsx`, end to end. It:

1. Runs the repository's own analyzer (`scripts/analyze_repair_strategies.py`) on the `R0` training run (`experiments/subset_train-r0`).
2. Reads the analyzer's `overall_summary.csv` and `mechanism_summary.csv`.
3. Derives the **common-core** metrics (shared by every strategy) and the **R0-specific** metrics.
4. Writes the exact table to `Train_results_R0.xlsx`.

Everything is read-only analysis of an existing run: **no experiments, no model calls, no benchmark scoring.** The outputs below are produced by actually executing the cells.

## Parameters — edit these if your run lives elsewhere

`STRATEGY` is this notebook's strategy. `RUN_DIR` is the **training run folder** to analyze. If a new/rerun `subset_train` experiment is saved under a different folder name, change `RUN_DIR` below and re-run the whole notebook — the two summary CSVs and the output `.xlsx` will regenerate from that run.

In [1]:
# === EDIT THESE if your training run is in a different folder ===
STRATEGY = 'R0'
RUN_DIR  = 'experiments/subset_train-r0'   # the subset_train run this notebook analyzes
# ================================================================

## Step 0 — Setup and run the analyzer
We run from the repository root and point the analyzer at this strategy's training run. The analyzer writes its tables under a scratch output directory.

In [2]:
import subprocess, sys, os, csv
from pathlib import Path

# Resolve the repository root (the folder that contains 'scripts/' and 'experiments/').
ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists(), 'Run this from inside the repo.'
NB_DIR = Path.cwd()  # folder containing this notebook (before we chdir to repo root)
os.chdir(ROOT)
PY = sys.executable  # the kernel's interpreter (the project venv)
assert 'STRATEGY' in globals() and 'RUN_DIR' in globals(), 'Run the Parameters cell first.'

# Analyzer output is written here, NEXT TO THIS NOTEBOOK (not a temp dir),
# so you can open overall_summary.csv / mechanism_summary.csv yourself.
OUT_DIR = str(NB_DIR / 'analysis_output')

assert Path(RUN_DIR).is_dir(), f'Missing run folder: {RUN_DIR}'
print('Repo root :', ROOT)
print('Strategy  :', STRATEGY)
print('Run folder:', RUN_DIR)
print('Analyzer output dir:', OUT_DIR)

Repo root : /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-
Strategy  : R0
Run folder: experiments/subset_train-r0
Analyzer output dir: /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output


In [3]:
# Run the repo's analyzer for this one strategy. This only reads the saved run.
cmd = [PY, 'scripts/analyze_repair_strategies.py',
       '--run', f'{STRATEGY}={RUN_DIR}',
       '--output-dir', OUT_DIR, '--no-figures']
print('$', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout.strip()[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'analyzer failed'
print('\nAnalyzer exit code:', res.returncode)

$ /Users/enruifung/.pyenv/versions/3.12.8/bin/python3 scripts/analyze_repair_strategies.py --run R0=experiments/subset_train-r0 --output-dir /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output --no-figures


R0: source=experiments/subset_train-r0, declared/evidenced queries=60, attempt queries=59, proposals=331, matrix cells=360

Analyzer exit code: 0


### Step 0b — Keep only the two summary CSVs
The analyzer emits many files we don't use (and an empty `figures/` folder). This removes everything except `overall_summary.csv` and `mechanism_summary.csv`, so re-running never leaves clutter behind.

In [4]:
# Keep the analyzer output tidy: the analyzer writes ~18 files (+ an empty
# figures/ dir) per strategy, but this notebook only needs the two summary
# CSVs. Delete everything else so the folder holds just what's useful.
import shutil
strat_out = Path(OUT_DIR) / STRATEGY
KEEP = {'overall_summary.csv', 'mechanism_summary.csv'}
removed = 0
for item in strat_out.iterdir():
    if item.name in KEEP:
        continue
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()
    removed += 1
print(f'Cleaned {strat_out}: removed {removed} extra item(s); kept {sorted(KEEP)}')
print('Files now:', sorted(p.name for p in strat_out.iterdir()))

Cleaned /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output/R0: removed 16 extra item(s); kept ['mechanism_summary.csv', 'overall_summary.csv']
Files now: ['mechanism_summary.csv', 'overall_summary.csv']


## Step 1 — Load the analyzer's summary tables

> **Where do these CSVs come from?** They are **not** pre-existing files in the
> repo. **Step 0 above created them** by running the analyzer. They are written to
> `train_result_analysis/analysis_output/<STRATEGY>/` (printed by Step 0). If you
> look before running Step 0, the folder will not exist yet — that's expected.

`overall_summary.csv` holds counts/means per run; `mechanism_summary.csv` holds the strategy-specific mechanism metrics.

In [5]:
def load_summary(path):
    d = {}
    with open(path) as f:
        for row in csv.DictReader(f):
            d[row['metric']] = row
    return d

overall = load_summary(f'{OUT_DIR}/{STRATEGY}/overall_summary.csv')
mech = load_summary(f'{OUT_DIR}/{STRATEGY}/mechanism_summary.csv')
print('overall_summary metrics:', len(overall))
print('mechanism_summary metrics:', len(mech))
print('\nmechanism metrics available:')
for k, v in mech.items():
    print(f'  {k}: count={v["count"]} valid_n={v["valid_n"]} rate={v["rate"]}')

overall_summary metrics: 85
mechanism_summary metrics: 4

mechanism metrics available:
  plan_scope_proposals: count=331 valid_n=331 rate=1
  four_or_more_measured_changed_slots: count=53 valid_n=285 rate=0.185964912280702
  multiple_declared_personal_targets: count=36 valid_n=331 rate=0.108761329305136
  commonsense_proposals_regressing_personal: count=15 valid_n=144 rate=0.104166666666667


## Step 2 — Common-core metrics (shared by every strategy)
These come straight from `overall_summary.csv`. 'Confirmed unchanged attempts' is shown as `N (pct%)` of total attempts, matching the R1 template already in the spreadsheet.

In [6]:
def iv(metric):
    return int(float(overall[metric]['value']))

queries              = iv('declared_queries')
repair_attempts      = iv('total_attempts')
personal_attempts    = iv('personal_attempts')
commonsense_attempts = iv('commonsense_attempts')
unchanged            = iv('unchanged_confirmed')
unchanged_pct        = round(unchanged / repair_attempts * 100, 2)
personal_fixes       = iv('personal_fixed_count_sum')
personal_regressions = iv('personal_newly_violated_count_sum')

core = [
    ('Queries', queries),
    ('Repair attempts', repair_attempts),
    ('Personal repair attempts', personal_attempts),
    ('Commonsense repair attempts', commonsense_attempts),
    ('Confirmed unchanged attempts', f'{unchanged} ({unchanged_pct}%)'),
    ('Personal constraint fixes', personal_fixes),
    ('Personal regressions', personal_regressions),
]
for k, v in core:
    print(f'  {k:40} {v}')

  Queries                                  60
  Repair attempts                          331
  Personal repair attempts                 187
  Commonsense repair attempts              144
  Confirmed unchanged attempts             222 (67.07%)
  Personal constraint fixes                14
  Personal regressions                     2


## Step 3 — R0-specific metrics
These describe R0's own mechanism, read from `mechanism_summary.csv`. Each is formatted `count / valid_n (rate%)`.

In [7]:
SPECIFIC = [('Plan-scope proposals (always whole-plan)', 'plan_scope_proposals', 'count_rate'), ('Proposals changing >=4 slots', 'four_or_more_measured_changed_slots', 'count_rate'), ('Attempts with multiple declared personal targets', 'multiple_declared_personal_targets', 'count_rate'), ('Commonsense proposals that regressed a personal constraint', 'commonsense_proposals_regressing_personal', 'count_rate')]

def fmt_count_rate(row):
    c = int(float(row['count']))
    n = int(float(row['valid_n']))
    r = float(row['rate']) * 100
    return f'{c} / {n} ({r:.1f}%)'

specific = []
for label, metric, kind in SPECIFIC:
    if metric not in mech:
        specific.append((label, 'n/a (metric not produced for this run)'))
        continue
    row = mech[metric]
    specific.append((label, fmt_count_rate(row) if kind == 'count_rate' else row['rate']))
for k, v in specific:
    print(f'  {k:60} {v}')

  Plan-scope proposals (always whole-plan)                     331 / 331 (100.0%)
  Proposals changing >=4 slots                                 53 / 285 (18.6%)
  Attempts with multiple declared personal targets             36 / 331 (10.9%)
  Commonsense proposals that regressed a personal constraint   15 / 144 (10.4%)


## Step 4 — Assemble the full table

In [8]:
rows = [('Metric', f'{STRATEGY} training result')]
rows += [(k, v) for k, v in core]
rows += [('', '')]  # spacer between core and strategy-specific
rows += [(k, v) for k, v in specific]

for k, v in rows:
    print(f'{k:60} | {v}')

Metric                                                       | R0 training result
Queries                                                      | 60
Repair attempts                                              | 331
Personal repair attempts                                     | 187
Commonsense repair attempts                                  | 144
Confirmed unchanged attempts                                 | 222 (67.07%)
Personal constraint fixes                                    | 14
Personal regressions                                         | 2
                                                             | 
Plan-scope proposals (always whole-plan)                     | 331 / 331 (100.0%)
Proposals changing >=4 slots                                 | 53 / 285 (18.6%)
Attempts with multiple declared personal targets             | 36 / 331 (10.9%)
Commonsense proposals that regressed a personal constraint   | 15 / 144 (10.4%)


## Step 5 — Write the exact table to `Train_results_R0.xlsx`

In [9]:
import openpyxl
from openpyxl.styles import Font

wb = openpyxl.Workbook()
ws = wb.active
ws.title = f'{STRATEGY} train results'

for r, (k, v) in enumerate(rows, start=1):
    ws.cell(r, 1, k)
    ws.cell(r, 2, v)

# bold the header row
ws.cell(1, 1).font = Font(bold=True)
ws.cell(1, 2).font = Font(bold=True)
ws.column_dimensions['A'].width = 52
ws.column_dimensions['B'].width = 40

out_xlsx = NB_DIR / f'Train_results_{STRATEGY}.xlsx'
wb.save(out_xlsx)
print('Wrote', out_xlsx)
print('Rows written:', len(rows))

Wrote /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/Train_results_R0.xlsx
Rows written: 13


## Step 6 — Read the file back to confirm it saved correctly

In [10]:
chk = openpyxl.load_workbook(out_xlsx, data_only=True)
cs = chk.active
print('Sheet:', cs.title, '| dims:', cs.dimensions)
print()
for row in cs.iter_rows(values_only=True):
    if any(c not in (None, '') for c in row):
        print(f'{str(row[0]):52} | {row[1]}')

Sheet: R0 train results | dims: A1:B13

Metric                                               | R0 training result
Queries                                              | 60
Repair attempts                                      | 331
Personal repair attempts                             | 187
Commonsense repair attempts                          | 144
Confirmed unchanged attempts                         | 222 (67.07%)
Personal constraint fixes                            | 14
Personal regressions                                 | 2
Plan-scope proposals (always whole-plan)             | 331 / 331 (100.0%)
Proposals changing >=4 slots                         | 53 / 285 (18.6%)
Attempts with multiple declared personal targets     | 36 / 331 (10.9%)
Commonsense proposals that regressed a personal constraint | 15 / 144 (10.4%)
